# Lezione 53 — Il dataset del capstone

> **Modulo:** capstone · **Tempo stimato:** 25 minuti
> **Prerequisiti:** Lezione 52 (architettura), Lezione 3 (train/val/test).
>
> Obiettivo pratico unico: caricare, **validare** e preparare il dataset di
> memorie che alimenta l'intero lab, riusando lo schema della Lezione 22.

## Teoria essenziale

Un sistema e' buono quanto i suoi dati. Prima di addestrare qualunque componente
ci assicuriamo che il dataset sia: **splittato** correttamente (train/val/test,
Lezione 3), **validato** contro lo schema (Lezione 22) e **pulito** dai casi
inutilizzabili (es. `type = unknown`). Registriamo un piccolo report di qualita'.

In [1]:
import pandas as pd
from pathlib import Path

proc = Path('..') / 'datasets' / 'processed'
train = pd.read_csv(proc / 'memory_train.csv')
val = pd.read_csv(proc / 'memory_val.csv')
test = pd.read_csv(proc / 'memory_test.csv')
print("righe:", {"train": len(train), "val": len(val), "test": len(test)})
print("tipi nel train:", train['type'].value_counts().to_dict())

righe: {'train': 213, 'val': 20, 'test': 15}
tipi nel train: {'episodic': 108, 'preference': 61, 'semantic': 43, 'unknown': 1}


In [2]:
TIPI = {"episodic", "semantic", "preference"}

def valida_e_pulisci(df):
    report = {"righe_iniziali": len(df)}
    # campi critici presenti (Lezione 22)
    df = df.dropna(subset=["memory_id", "text"]).copy()
    # scarto i tipi non validi (es. 'unknown')
    prima = len(df)
    df = df[df["type"].isin(TIPI)].copy()
    report["scartati_tipo"] = prima - len(df)
    # importance nel range
    df["importance"] = df["importance"].clip(0.0, 1.0)
    report["righe_finali"] = len(df)
    return df, report

train_c, rep_tr = valida_e_pulisci(train)
val_c, _ = valida_e_pulisci(val)
test_c, _ = valida_e_pulisci(test)
print("report train:", rep_tr)

report train: {'righe_iniziali': 213, 'scartati_tipo': 1, 'righe_finali': 212}


Leggi il report: le righe con `type` non valido vengono scartate, i campi
critici garantiti, l'importanza riportata nel range. Ora il dataset e' pronto per
i componenti.

In [3]:
# controlli di non-regressione sul dataset pulito
assert train_c['type'].isin(TIPI).all()
assert not train_c[['memory_id', 'text']].isna().any().any()
assert train_c['importance'].between(0, 1).all()
# nessuna fuga di dati grossolana: gli id di test non sono nel train
assert set(test_c['memory_id']).isdisjoint(set(train_c['memory_id']))
print("dataset validato: train", len(train_c), "| val", len(val_c), "| test", len(test_c))
print("nessun id di test presente nel train (no leakage)  OK")

dataset validato: train 212 | val 19 | test 14
nessun id di test presente nel train (no leakage)  OK


## Riepilogo (max 8 punti)

1. Un sistema vale quanto i suoi dati: prima si valida il dataset.
2. Split train/val/test (Lezione 3).
3. Validazione contro lo schema (Lezione 22): campi critici garantiti.
4. Si scartano i casi inutilizzabili (`type = unknown`).
5. L'importanza si riporta nel range [0,1].
6. Un report di qualita' documenta cosa e' stato scartato.
7. Si verifica l'assenza di leakage tra split.
8. Il dataset pulito alimenta tutti i componenti.

## Quiz

1. Perche' scartare le righe con `type = unknown`?
2. Cosa verifica il controllo sugli id di test?
3. Perche' un report di qualita' e' utile anche se "i dati sembrano ok"?

*(Risposte: 1. non offrono un'etichetta valida per il classificatore; 2. che non
ci sia leakage — memorie di test finite nel train; 3. rende esplicito e
riproducibile cosa e' stato rimosso, evitando sorprese a valle.)*